In [1]:
import numpy as np
import pandas as pd

def OHspecial_transform(df):
    names = df.columns.values
    data = df.values
    labels = []
    
    for j in range(len(names)):
        col_data = data[:, j]
        temp = np.unique(col_data[~np.isnan(col_data)])
        temp = temp[temp >= 0]
        
        if len(temp) == 0:
            # All data in column j is NaN or negative
            var = np.full((data.shape[0], 1), np.nan)
            label = [names[j]]
        elif len(temp) <= 2:
            # Keep original data
            var = col_data.reshape(-1, 1)
            label = [names[j]]
        else:
            # One-hot encode
            var = np.zeros((data.shape[0], len(temp)))
            for l in range(len(temp)):
                var[:, l] = np.where(col_data == temp[l], 1, 0)
            idx = np.where(np.isnan(col_data))[0]
            var[idx, :] = np.nan
            label = [f"{names[j]}_{int(temp[l])}" for l in range(len(temp))]
        
        if j == 0:
            OH = var
        else:
            OH = np.hstack((OH, var))
        labels.extend(label)
    
    OH_df = pd.DataFrame(OH, columns=labels)
    return OH_df


import numpy as np

import matplotlib.pyplot as plt
import time
import sys
sys.path.append('../pca_full')
from pca_full import pca_full

# Ensure reproducibility
np.random.seed(42)

# Set default text interpreter to LaTeX (for matplotlib)
plt.rcParams['text.usetex'] = True

# Read in data and remove non-cultural features + features missing > 50% of possible values

# From EA
KinOrgRaw = pd.read_csv('KinshipOrgDataRaw.csv')
SubRaw = pd.read_csv('SubsistenceDataRaw.csv')
# EA_All_Raw = pd.read_csv('EA_All.csv')  # Commented out in MATLAB code

# From Pulotu
IsoRaw = pd.read_csv('IsoDataRaw.csv')
ReligRaw = pd.read_csv('RelDataRaw.csv')

# Other datasets
Binford = pd.read_csv('Binford_all.csv')
Seshat = pd.read_csv('Seshat.csv')
Birds = pd.read_csv('data_birds_OH.csv')

KinOrg = OHspecial_transform(KinOrgRaw)
Sub = OHspecial_transform(SubRaw)
Iso = OHspecial_transform(IsoRaw)
Relig = OHspecial_transform(ReligRaw)

for w in range(4, 5):  # MATLAB for w = 7:7

    if w == 1:
        X = KinOrg
    elif w == 2:
        X = Sub
    elif w == 3:
        X = Relig
    elif w == 4:
        X = Iso
    elif w == 5:
        X = Binford
    elif w == 6:
        X = Seshat
    elif w == 7:
        X = Birds

    # Remove features missing > 50% of their entries
    dim = X.shape
    X = X.iloc[:, 0:dim[1]]  # Redundant but kept for similarity
    cols = X.columns.values
    X = X.to_numpy()

    counts = []
    for j in range(dim[1]):
        temp = np.isnan(X[:, j]).astype(int)
        idx = np.where(temp == 1)[0]
        counts.append(len(idx))

    counts = np.array(counts) / dim[0]
    idx = np.where(counts > 0.5)[0]
    cols = np.delete(cols, idx)
    X = np.delete(X, idx, axis=1)
    dim = X.shape

    # Compute initial mean and standard deviation, ignoring NaNs
    MnInit = np.tile(np.nanmean(X, axis=0), (dim[0], 1))
    STDInit = np.tile(np.nanstd(X, axis=0), (dim[0], 1))

    if w < 5:
        X_ = X - MnInit
    elif w == 5:
        X_ = X
    elif w == 6:
        X_ = (X - MnInit) / STDInit
    elif w == 7:
        X_ = X

    X_ = X_.T

    MaxStop = min(dim)
    if MaxStop > 100:
        MaxStop = 100
    zz = np.arange(2, MaxStop + 1)
    rms = np.zeros(MaxStop - 1)
    accu = np.zeros(MaxStop - 1)
    rmsA = np.zeros((MaxStop - 1, 80))  # Assuming maxiters is 80
    costA = np.zeros((MaxStop - 1, 80))
    var_exp = np.zeros((MaxStop - 1, MaxStop))

    for y in range(MaxStop - 1):

        z = zz[y]
        print('###############################')
        print(f'number of PCs: {z}')
        print('###############################')
        if y > 0:
            time.sleep(1)

        for k in range(1):  # Change if you want to do replications

            opts = {
                'maxiters': 80,
                'algorithm': 'vb',
                'uniquesv': 0,
                'rmsstop': [80, np.finfo(float).eps, np.finfo(float).eps],
                'cfstop': [80, 0, 0],
                'minangle': 0
            }

            A, S, Mu, V, cv, hp, lc = pca_full(X_, z, **opts)

            if z == 20:
                plt.figure()
                plt.scatter(S[0, :], S[1, :])
                plt.show()

            print('Learning is finished.')

            Xrec = Mu[:, np.newaxis] + A @ S

            accu_num = np.abs(np.round(Xrec.T + MnInit) - X)
            accu_num = len(np.where(accu_num > 0)[0])
            accu[y] = 1 - accu_num / len(np.where(~np.isnan(X_.flatten()))[0])
            rms[y] = lc['rms'][-1]
            rmsA[y, :] = lc['rms']
            costA[y, :] = lc['cost']

            vv = np.zeros(MaxStop)
            v = np.linalg.eigvals(np.cov(Xrec))
            v = np.flipud(np.sort(np.real(v)))
            v = v[:MaxStop]
            v = np.round(v, 4)
            v = v / np.sum(v)
            vv[:len(v)] = v
            var_exp[y, :] = vv

        if y > 0 and (rms[y - 1] - rms[y]) > 0:
            break

    # Since w == 7
    AFin, SFin, MuFin, VFin, cvFin, hpFin, lcFin = pca_full(X_, zz[y - 1], **opts)
    Xrec = MuFin[:, np.newaxis] + AFin @ SFin
    Vr = np.zeros_like(X_)
    for i in range(X_.shape[0]):
        for j in range(X_.shape[1]):
            Vr[i, j] = (AFin[i, :] @ cvFin['S'][j] @ AFin[i, :].T +
                        SFin[:, j].T @ cvFin['A'][i] @ SFin[:, j] +
                        np.sum(np.sum(cvFin['S'][j] * cvFin['A'][i])) +
                        cvFin['Mu'][i])
    VrFin = Vr
    T = pd.DataFrame(Xrec.T, columns=cols)
    T.to_csv('Finch_VBPCA_Recon_All.csv', index=False)
    XrecFin = Xrec
    rmsFin = rms
    rmsAFin = rmsA
    costAFin = costA
    var_exp_Fin = var_exp
    accu_Fin = accu
    plt.figure()
    plt.scatter(SFin[0, :], SFin[1, :], c='b', marker='o')
    plt.title('Isolation')
    plt.show()

    # Clear variables
    rmsA = None
    costA = None
    var_exp = None
    Vr = None
    accu = None


###############################
number of PCs: 2
###############################
No empty rows or columns
Shape of XA1: (18, 137)
subtract mu function
Shape of XB1: (18, 137)
Mu is : [ 1.65704929e-17 -9.29501086e-18  5.30255773e-17 -2.94377317e-17
  3.88998598e-17  1.08288799e-17  2.45705096e-17  5.64211701e-17
 -5.73311890e-17  2.86655945e-17 -7.23327122e-17  2.01858732e-17
  2.79658451e-17  5.88754634e-17 -2.56528805e-17 -4.58387537e-17
  0.00000000e+00  3.53252781e-17]
M is [[ True  True  True ...  True  True  True]
 [ True  True  True ...  True  True  True]
 [ True  True  True ...  True  True  True]
 ...
 [ True  True  True ...  True  True  True]
 [ True  True  True ...  True  True  True]
 [ True  True  True ...  True  True  True]]
Shape of XB3: (18, 137)
Shape of XA2: (18, 137)
Step 0: rms = 0.501331
subtract mu function
Shape of XB1: (18, 137)
Mu is : [-0.01047914 -0.017101    0.00554384 -0.03129787  0.01090814  0.00868379
  0.00823979  0.01574975  0.00586788  0.01414607 -0.01558

ValueError: The truth value of an array with more than one element is ambiguous. Use a.any() or a.all()